In [9]:
import sys
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
sys.path.append(os.path.abspath(".."))

In [19]:
from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.utils import load_jsonl, save_responses_to_jsonl

In [29]:
def map_text_entities_to_offsets(samples):
    """
    Convierte las entidades basándose en 'text' a entidades con 'start' y 'end'.
    """
    processed_samples = []

    for sample in samples:
        text = sample.get("text", "")
        entities = sample.get("entities", [])
        mapped_entities = []

        for ent in entities:
            # Si ya tiene 'start' y 'end', los conservamos
            if "start" in ent and "end" in ent:
                mapped_entities.append(ent)
                continue

            ent_text = ent.get("text", "")
            label = ent.get("label", "")

            if not ent_text:
                continue

            # Buscar la posición exacta del texto dentro de la muestra
            start = text.find(ent_text)
            if start != -1:
                end = start + len(ent_text)
                mapped_entities.append({
                    "start": start,
                    "end": end,
                    "label": label,
                    "text": ent_text
                })

        processed_samples.append({
            "id": sample.get("id"),
            "text": text,
            "entities": mapped_entities
        })

    return processed_samples

In [30]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
OUTPUT_PATH = '../results/llm/predictions.jsonl'
FORMATTED_OUTPUT_PATH = '../results/llm/predictions_formatted.jsonl'
gold_samples = load_jsonl(VALIDATION_PATH)
llm_predictions = load_jsonl(OUTPUT_PATH)
formatted_predictions = map_text_entities_to_offsets(llm_predictions)
save_responses_to_jsonl(formatted_predictions, FORMATTED_OUTPUT_PATH)

In [31]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, FORMATTED_OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, FORMATTED_OUTPUT_PATH)


STRICT EVALUATION
Micro -> P: 0.69 | R: 0.44 | F1: 0.54 | TP: 18 | FP: 8 | FN: 23

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.67   0.20   0.31       2       1       8
ID_CARD_NUMBER              1.00   0.50   0.67       2       0       2
ADDRESS                     1.00   0.50   0.67       2       0       2
REGISTRATION_NUMBER         0.33   0.20   0.25       1       2       4
DATE                        0.62   0.62   0.62       5       3       3
AMOUNT                      0.75   0.75   0.75       3       1       1
ORGANIZATION                0.75   0.50   0.60       3       1       3

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 41 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------